# Q

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd

import folium
from folium import Choropleth
from folium.features import GeoJson, GeoJsonTooltip

import branca
import branca.colormap as cm

import ipywidgets as widgets
from IPython.display import display, clear_output

In [2]:
aq = pd.read_csv('/Users/chintanjikkar/Desktop/PACE/Fall 25/Visual Analytics/Team Project/Processed data/air_quality_processed_with_annual_2024_2025.csv')

In [3]:
# 0. Make a working copy
aq_full = aq.copy()

# 1. Basic structure checks
print("Shape:", aq_full.shape)
print("\nColumns:", list(aq_full.columns))
print("\nDtypes:\n", aq_full.dtypes)

# 2. Check expected columns exist
expected_cols = {
    "Geo Join ID",
    "Geo Place Name",
    "Geo Type Name",
    "Start_Date",
    "Time Period",
    "Name",
    "Measure Info",
    "Data Value"
}

missing_cols = expected_cols - set(aq_full.columns)
extra_cols = set(aq_full.columns) - expected_cols

print("\nMissing expected columns:", missing_cols)
print("Extra columns (ok, just be aware):", extra_cols)

# 3. Rename to cleaner schema we used earlier
aq_full = aq_full.rename(columns={
    "Geo Join ID": "geo_join_id",
    "Geo Place Name": "geo_place_name",
    "Geo Type Name": "geo_type_name",
    "Start_Date": "start_date",
    "Time Period": "time_period",
    "Name": "pollutant_name",
    "Measure Info": "measure_info",
    "Data Value": "data_value"
})

# 4. Parse dates & derive year
aq_full["start_date"] = pd.to_datetime(aq_full["start_date"], errors="coerce")

# Prefer year from time_period (it encodes 2009 Annual, 2017 Q1, etc.)
aq_full["year_from_period"] = (
    aq_full["time_period"]
    .str.extract(r'(\d{4})', expand=False)
    .astype(float)
)

# Fallback: if year_from_period is missing, use start_date
mask_missing_year = aq_full["year_from_period"].isna() & aq_full["start_date"].notna()
aq_full.loc[mask_missing_year, "year_from_period"] = aq_full.loc[mask_missing_year, "start_date"].dt.year

aq_full["year_from_period"] = aq_full["year_from_period"].astype("Int64")

print("\nUnique years (year_from_period):", sorted(aq_full["year_from_period"].dropna().unique().tolist()))

# 5. Check that 2024 & 2025 are actually present
for y in [2024, 2025]:
    count_y = (aq_full["year_from_period"] == y).sum()
    print(f"Rows with year {y}: {count_y}")

# 6. Quick category sanity checks
print("\nGeo type breakdown:\n", aq_full["geo_type_name"].value_counts())

print("\nSample of pollutants:")
print(aq_full["pollutant_name"].value_counts().head(15))

print("\nSample of time_period values:")
print(aq_full["time_period"].dropna().unique()[:10])

# 7. Missing values summary
print("\nMissing values per column:\n", aq_full.isna().sum())

# 8. Duplicates check on a key combination
dup_keys = ["geo_join_id", "geo_place_name", "geo_type_name", "time_period", "pollutant_name", "measure_info"]
if all(col in aq_full.columns for col in dup_keys):
    dup_count = aq_full.duplicated(subset=dup_keys).sum()
    print(f"\nPotential duplicate rows on key {dup_keys}: {dup_count}")
else:
    print("\nSkipping duplicate check: some key columns missing")

# 9. Sanity check on data_value
print("\nData value describe():\n", aq_full["data_value"].describe())

neg_values = (aq_full["data_value"] < 0).sum()
print("Negative data_value count:", neg_values)

extreme_high = (aq_full["data_value"] > aq_full['data_value'].quantile(0.99)).sum()
print("Values above 99th percentile:", extreme_high)

# 10. Build BORO-level subset for visualizations (Borough only)

aq_boro = aq_full[aq_full["geo_type_name"] == "Borough"].copy()
print("\nBorough-level rows:", aq_boro.shape[0])

# Map borough names to match crime project naming
boro_map = {
    "Bronx": "BRONX",
    "Brooklyn": "BROOKLYN",
    "Manhattan": "MANHATTAN",
    "Queens": "QUEENS",
    "Staten Island": "STATEN ISLAND"
}

aq_boro["BORO_NM_IMPUTED"] = aq_boro["geo_place_name"].map(boro_map)
missing_boro_names = aq_boro["BORO_NM_IMPUTED"].isna().sum()
print("Rows with unmapped borough names:", missing_boro_names)

print("\nSample BORO_NM_IMPUTED:", aq_boro["BORO_NM_IMPUTED"].value_counts())

# 11. Check year x borough coverage for sanity
coverage = (
    aq_boro
    .groupby(["year_from_period", "BORO_NM_IMPUTED"])
    .size()
    .unstack(fill_value=0)
)

print("\nYear x Borough coverage table (counts of rows):\n", coverage.tail(10))


Shape: (19426, 9)

Columns: ['Unnamed: 0', 'Geo Join ID', 'Geo Place Name', 'Geo Type Name', 'Start_Date', 'Time Period', 'Name', 'Measure Info', 'Data Value']

Dtypes:
 Unnamed: 0          int64
Geo Join ID         int64
Geo Place Name     object
Geo Type Name      object
Start_Date         object
Time Period        object
Name               object
Measure Info       object
Data Value        float64
dtype: object

Missing expected columns: set()
Extra columns (ok, just be aware): {'Unnamed: 0'}

Unique years (year_from_period): [2005, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Rows with year 2024: 282
Rows with year 2025: 282

Geo type breakdown:
 geo_type_name
UHF42       7560
CD          7080
UHF34       3706
Borough      900
Citywide     180
Name: count, dtype: int64

Sample of pollutants:
pollutant_name
Nitrogen dioxide (NO2)                                    6627
Fine particles (PM 2.5)                             

In [6]:
# =========================================
# 1. Start from your current df `aq`
# =========================================

aq_full = aq.copy()

print("Original columns:", aq_full.columns.tolist())

# Rename to cleaner, lowercase schema (like in your sanity check)
aq_full = aq_full.rename(columns={
    "Geo Join ID": "geo_join_id",
    "Geo Place Name": "geo_place_name",
    "Geo Type Name": "geo_type_name",
    "Start_Date": "start_date",
    "Time Period": "time_period",
    "Name": "pollutant_name",
    "Measure Info": "measure_info",
    "Data Value": "data_value"
})

# You have an extra index column; optional to drop it
if "Unnamed: 0" in aq_full.columns:
    aq_full = aq_full.drop(columns=["Unnamed: 0"])

# =========================================
# 2. Ensure year_from_period exists
# =========================================

if "year_from_period" not in aq_full.columns:
    aq_full["year_from_period"] = (
        aq_full["time_period"]
        .str.extract(r"(\d{4})", expand=False)
        .astype("Int64")
    )

print("Years in data:", sorted(aq_full["year_from_period"].dropna().unique().tolist()))

# =========================================
# 3. Create BORO_NM_IMPUTED for borough rows
# =========================================

boro_map = {
    "Bronx": "BRONX",
    "Brooklyn": "BROOKLYN",
    "Manhattan": "MANHATTAN",
    "Queens": "QUEENS",
    "Staten Island": "STATEN ISLAND"
}

# Only borough rows have a meaningful borough; others will be NaN
aq_full["BORO_NM_IMPUTED"] = np.where(
    aq_full["geo_type_name"] == "Borough",
    aq_full["geo_place_name"].map(boro_map),
    np.nan
)

print("\nBORO_NM_IMPUTED value counts (including NaN):")
print(aq_full["BORO_NM_IMPUTED"].value_counts(dropna=False))

# =========================================
# 4. Build composite from Tier 1 pollutants (PM2.5, NO2, O3)
# =========================================

tier1_pollutants = [
    "Fine particles (PM 2.5)",
    "Nitrogen dioxide (NO2)",
    "Ozone (O3)"
]

tier1 = aq_full[
    (aq_full["geo_type_name"] == "Borough") &
    (aq_full["pollutant_name"].isin(tier1_pollutants))
].copy()

# Pivot: Borough × Year × Pollutant
tier1_pivot = tier1.pivot_table(
    index=["BORO_NM_IMPUTED", "year_from_period"],
    columns="pollutant_name",
    values="data_value",
    aggfunc="mean"
)

tier1_pivot = tier1_pivot.rename(columns={
    "Fine particles (PM 2.5)": "PM25",
    "Nitrogen dioxide (NO2)": "NO2",
    "Ozone (O3)": "O3"
})

# Compute composite (row-wise mean across 3 pollutants)
tier1_pivot["AQI_composite"] = tier1_pivot[["PM25", "NO2", "O3"]].mean(axis=1)

print("\nComposite pivot head:")
print(tier1_pivot.head())

# =========================================
# 5. Merge composite back into FULL dataset (keep ALL pollutants)
# =========================================

aq_with_comp = aq_full.merge(
    tier1_pivot[["AQI_composite"]],
    on=["BORO_NM_IMPUTED", "year_from_period"],
    how="left"
)

print("\nFinal shape with composite:", aq_with_comp.shape)
print(aq_with_comp[["geo_type_name", "geo_place_name", "year_from_period",
                    "pollutant_name", "data_value", "AQI_composite"]].head(10))

Original columns: ['Unnamed: 0', 'Geo Join ID', 'Geo Place Name', 'Geo Type Name', 'Start_Date', 'Time Period', 'Name', 'Measure Info', 'Data Value']
Years in data: [2005, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

BORO_NM_IMPUTED value counts (including NaN):
BORO_NM_IMPUTED
NaN              18526
BRONX              180
BROOKLYN           180
MANHATTAN          180
QUEENS             180
STATEN ISLAND      180
Name: count, dtype: int64

Composite pivot head:
pollutant_name                         PM25        NO2     O3  AQI_composite
BORO_NM_IMPUTED year_from_period                                            
BRONX           2008              14.080000  29.290000    NaN      21.685000
                2009              10.910000  23.246667  25.30      19.818889
                2010              12.036667  23.256667  33.16      22.817778
                2011              10.803333  21.506667  32.91      21.740000
        

In [7]:
# ============================================================
# Add Composite AQI Column to Existing Dataset (KEEP EVERYTHING)
# ============================================================

# 1. Make a working copy
aq_with_comp = aq_full.copy()   # or aq if you prefer

# 2. Identify Tier 1 pollutants
tier1_map = {
    "Fine particles (PM 2.5)": "PM25",
    "Nitrogen dioxide (NO2)": "NO2",
    "Ozone (O3)": "O3"
}

# 3. Keep Borough-level rows for composite calculation
boro_only = aq_with_comp[aq_with_comp["geo_type_name"] == "Borough"].copy()

# 4. Extract Tier 1 pollutants (PM2.5, NO2, O3)
tier1 = boro_only[boro_only["pollutant_name"].isin(tier1_map.keys())].copy()

# 5. Pivot these into PM25, NO2, O3 columns for composite calc
tier1_pivot = tier1.pivot_table(
    index=["BORO_NM_IMPUTED", "year_from_period"],
    columns="pollutant_name",
    values="data_value",
    aggfunc="mean"
).rename(columns=tier1_map)

# 6. Compute Composite AQI = mean of PM25, NO2, O3
tier1_pivot["AQI_composite"] = tier1_pivot[["PM25", "NO2", "O3"]].mean(axis=1)

# 7. Merge this composite column BACK into full dataset (keep all pollutants)
aq_with_comp = aq_with_comp.merge(
    tier1_pivot["AQI_composite"],
    on=["BORO_NM_IMPUTED", "year_from_period"],
    how="left"
)

# 8. Save final dataset
aq_with_comp.to_csv("aq_full_with_composite.csv", index=False)

print("Final dataset saved: aq_full_with_composite.csv")
print("New column added: AQI_composite")
print("Shape:", aq_with_comp.shape)
print(aq_with_comp.head())

Final dataset saved: aq_full_with_composite.csv
New column added: AQI_composite
Shape: (19426, 11)
   geo_join_id                      geo_place_name geo_type_name  start_date  \
0          310  Bay Ridge and Dyker Heights (CD10)            CD  2005-01-01   
1          310  Bay Ridge and Dyker Heights (CD10)            CD  2005-01-01   
2          310  Bay Ridge and Dyker Heights (CD10)            CD  2005-01-01   
3          404               Bayside - Little Neck         UHF42  2005-01-01   
4          404               Bayside - Little Neck         UHF42  2005-01-01   

  time_period                          pollutant_name     measure_info  \
0        2005           Annual vehicle miles traveled  per square mile   
1        2005    Annual vehicle miles traveled (cars)  per square mile   
2        2005  Annual vehicle miles traveled (trucks)  per square mile   
3        2005           Annual vehicle miles traveled  per square mile   
4        2005            Outdoor Air Toxics - Benz